# Anchor separation against prompt length

One panel per model. Each point is a **variant** - one template activation cache, the
`file` column of that model's `surface/rows.parquet`, 41 per model.

- **y** is the **geodesic distance** between that variant's `very_low` centroid and its
  `catastrophic` centroid, measured *along the fitted stakes surface* rather than through
  the ambient PLS space.
- **x** is the **mean token length** of that variant's prompts, counted with the model's
  own tokenizer on the chat-formatted text the activation cache was built from.

The question the panels answer is whether the ladder's span is just a restatement of how
long the prompts are. A variant is the right unit for that: register and template fix both
the wording length and the cache the centroids come from, so length varies between variants
and barely within one.

### Why a geodesic, and how it is computed

`arc_length_parallel` and `arc_length_orthogonal` are *slice* coordinates, not orthogonal
geodesic ones - `scripts/stakes_height_slices.py` says so in its first line, and the README
repeats that `arc_length_orthogonal` is snapped PLS3 height and not a length. Differencing
either column between two centroids therefore does not give a distance on the surface, and
Pythagoras on the pair would give one only if the coordinate system were orthonormal, which
it is not.

So the distance is solved for directly. The saved surface is a graph over the
(PLS1, PLS3) plane,

```
S(u, v) = (u, f(u, v), v)        f = the middle column of SliceSurface.evaluate
```

embedded in the ambient PLS1-PLS3 score space. A path on the surface is a path in `(u, v)`,
and its length is the length of its image under `S`. The geodesic is found by discretizing
the path into `GEODESIC_NODES` points with the two endpoints pinned, then minimizing the
**discrete energy** `sum ||S(p[i+1]) - S(p[i])||^2` over the interior points. Energy rather
than length: both are minimized by the same curve, but length is invariant to
reparameterization and so has no isolated minimizer, while energy pins the nodes to equal
arc spacing and makes the problem well posed. The reported distance is the polyline length
`sum ||S(p[i+1]) - S(p[i])||` of the converged path.

The gradient is analytic. `S` is a graph, so its Jacobian is `[[1, 0], [f_u, f_v], [0, 1]]`
and the energy gradient at an interior node is `2 J' (2 S[i] - S[i-1] - S[i+1])`.
`surface_embedding` returns `f_u` and `f_v` in closed form from the same polynomial
coefficients `evaluate` uses, and the cell below checks both against the pipeline's own
function.

Three properties make the result checkable rather than merely plausible, and all three are
verified further down: on a flat surface the solver must return the straight-line distance;
refining the node count must not move the answer; and the geodesic can never come out
shorter than the ambient chord between the same two points.

### Centroids

A centroid is the mean of a class's rows *within one variant*, taken in the surface
parameters `(surface_u, surface_v)` - the same per-variant class mean
`notebooks/arc_length_normalization.ipynb` anchors on, just kept as a position on the
surface instead of collapsed onto the parallel coordinate. Taking the mean in surface
parameters keeps the centroid on the surface by construction, so the geodesic runs between
two points that actually lie on it.

`catastrophic` is the high anchor rather than `existential` for the reason the
normalization notebook gives: the existential rows are the noisy end of the ladder and the
one place the models disagree about ordering, with `gemma-4-31B-it` placing `existential`
below `catastrophic`. `very_low` is the lowest class in every model's own ordering.

### Reading the panels

Each panel carries its own y scale. Geodesic length is in that model's unscaled PLS score
units, which are not comparable between models - only the *shape* of each relationship is.
The x scales are comparable across the four Qwen models, which share a tokenizer, but not
with `gemma-4-31B-it`.

## Configuration

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from numpy.polynomial import polynomial as poly
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.stats import pearsonr, spearmanr
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'scripts' / 'stakes_surface_pipeline.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.stakes_surface_bundle import load_surface_bundle

ARTIFACT_ROOT = ROOT / 'artifacts' / 'content' / 'artifacts'

LOW_STAKES_CLASS = 'very_low'          # anchors the near end of the geodesic
HIGH_STAKES_CLASS = 'catastrophic'     # the far end; existential is the noisier class
MODELS = None                          # None -> every model directory under ARTIFACT_ROOT

GEODESIC_NODES = 65                    # path nodes, both fixed endpoints included
REFINEMENT_NODES = 129                 # re-solved independently as a convergence check
SOLVER_TOLERANCE = 1e-10

# Token counts come from the model's own tokenizer. 'chat' counts the chat-formatted
# string the activation cache was built from - what the model actually read - and 'bare'
# counts the prompt text alone. The chat template adds a constant per model, so the two
# differ by a shift and rank the variants identically.
TOKEN_TEXT = 'chat'                    # 'chat' or 'bare'
TOKEN_SCOPE = 'all'                    # 'all' rows in the variant, or 'anchors' only
TOKEN_MATCH_BIN = 1                    # token-length bin width for the matched control; 1 is exact
HF_CACHE_DIR = None                    # None -> the Hugging Face default location

WRITE_TABLE = False                    # write per-model CSVs beside each surface model
TABLE_NAME = 'anchor_geodesic_vs_token_length.csv'

# Column and path names shared with scripts/stakes_surface_pipeline.py.
VARIANT, STAKES, TEXT = 'file', 'stakes', 'text'
SURFACE_U, SURFACE_V = 'surface_u', 'surface_v'
TARGET = 'arc_length_parallel'             # the pipeline's class-ordering statistic
TRAINING_ROWS = Path('surface') / 'rows.parquet'

if TOKEN_TEXT not in ('chat', 'bare'):
    raise ValueError("TOKEN_TEXT must be 'chat' or 'bare'.")
if TOKEN_SCOPE not in ('all', 'anchors'):
    raise ValueError("TOKEN_SCOPE must be 'all' or 'anchors'.")
if not (isinstance(TOKEN_MATCH_BIN, int) and TOKEN_MATCH_BIN >= 1):
    raise ValueError('TOKEN_MATCH_BIN must be a positive whole number of tokens.')

print(f'Repository root: {ROOT}')
print(f'Artifacts:       {ARTIFACT_ROOT}')
print(f'Anchors:         {LOW_STAKES_CLASS} -> {HIGH_STAKES_CLASS}')
print(f'Path nodes:      {GEODESIC_NODES} (refinement check at {REFINEMENT_NODES})')
print(f'Token counts:    {TOKEN_TEXT}-formatted text, averaged over {TOKEN_SCOPE} rows')

Repository root: c:\Users\91967\Desktop\AISC\stakes_manifold
Artifacts:       c:\Users\91967\Desktop\AISC\stakes_manifold\artifacts\content\artifacts
Anchors:         very_low -> catastrophic
Path nodes:      65 (refinement check at 129)
Token counts:    chat-formatted text, averaged over all rows


## The surface, its derivatives, and the geodesic solver

`surface_embedding` reproduces `SliceSurface.evaluate` and adds the two partial derivatives
of the height, both in closed form from the same cubic coefficients. Outside the fitted
PLS1 range the pipeline extends the surface linearly in `t`, so the height's `u`-derivative
there is the slope at the clipped endpoint; the expression below handles that case the way
`evaluate` does, so the two agree to machine precision rather than merely closely - the
check below reports the gap, which is a rounding residue of order 1e-15.

In [2]:
def surface_embedding(surface, uv):
    """Points on the saved surface, and the height's two partial derivatives.

    Returns `(points, gradient)` where `points` are ambient (PLS1, PLS2, PLS3) rows
    matching `SliceSurface.evaluate`, and `gradient` holds (df/du, df/dv) - enough to
    build the Jacobian of the graph map, which is [[1, 0], [f_u, f_v], [0, 1]].
    """
    uv = np.atleast_2d(np.asarray(uv, dtype=np.float64))
    u, v = uv[:, 0], uv[:, 1]
    t = (u - surface.center[0]) / surface.scale[0]
    clipped = np.clip(t, *surface.t_bounds)
    z = (v - surface.center[1]) / surface.scale[1]
    # Each row of `coefficients` is a cubic in z giving one coefficient of the slice curve.
    curve = np.stack([poly.polyval(z, row) for row in surface.coefficients], axis=-1)
    slope = np.stack([poly.polyval(z, poly.polyder(row)) for row in surface.coefficients], axis=-1)
    powers = np.stack([clipped ** k for k in range(4)], axis=-1)
    derivatives = np.stack([k * clipped ** max(k - 1, 0) for k in range(4)], axis=-1)
    overhang = t - clipped                       # nonzero only on the linear extension
    height = (curve * powers).sum(1) + overhang * (curve * derivatives).sum(1)
    gradient = np.column_stack([
        (curve * derivatives).sum(1) / surface.scale[0],
        ((slope * powers).sum(1) + overhang * (slope * derivatives).sum(1)) / surface.scale[1],
    ])
    return np.column_stack([u, height, v]), gradient


def geodesic(surface, start, end, nodes=None, tolerance=None):
    """Shortest path along the surface between two points of its (u, v) domain.

    The straight segment in `(u, v)` is the initial guess; the interior nodes then
    minimize the discrete energy. Returns the path length, the ambient chord length
    between the endpoints, and the converged path in surface parameters.
    """
    nodes = GEODESIC_NODES if nodes is None else nodes
    tolerance = SOLVER_TOLERANCE if tolerance is None else tolerance
    if nodes < 3:
        raise ValueError('A path needs at least one interior node.')
    start, end = np.asarray(start, dtype=np.float64), np.asarray(end, dtype=np.float64)
    path = start + np.linspace(0.0, 1.0, nodes)[:, None] * (end - start)

    def energy_and_gradient(interior):
        path[1:-1] = interior.reshape(-1, 2)
        points, height_gradient = surface_embedding(surface, path)
        pull = 2 * points[1:-1] - points[:-2] - points[2:]
        gradient = 2 * np.column_stack([pull[:, 0] + height_gradient[1:-1, 0] * pull[:, 1],
                                        pull[:, 2] + height_gradient[1:-1, 1] * pull[:, 1]])
        return float((np.diff(points, axis=0) ** 2).sum()), gradient.ravel()

    result = minimize(energy_and_gradient, path[1:-1].ravel(), jac=True, method='L-BFGS-B',
                      options=dict(maxiter=10000, maxfun=10000, ftol=tolerance, gtol=tolerance))
    if not result.success:
        raise RuntimeError(f'Geodesic solver did not converge: {result.message}')
    path[1:-1] = result.x.reshape(-1, 2)
    points, _ = surface_embedding(surface, path)
    length = float(np.linalg.norm(np.diff(points, axis=0), axis=1).sum())
    chord = float(np.linalg.norm(points[-1] - points[0]))
    if length < chord - 1e-9:
        raise RuntimeError('A surface path came out shorter than the ambient chord.')
    return length, chord, path

### Does the solver do what it claims?

Two checks that do not depend on any model's data. The embedding must agree with the
pipeline's own `evaluate` and with a central difference of it; and on a surface with every
coefficient zeroed - a plane - the geodesic must come back as the straight line. The first
and third are exact statements about the mathematics, so they are held to machine precision
rather than to a modelling tolerance; the second is a finite difference and carries its own
step error.

In [3]:
def solver_checks(surface, samples=400, step=1e-6, seed=0):
    generator = np.random.default_rng(seed)
    spread = np.column_stack([
        generator.uniform(*(np.asarray(surface.x_bounds) * 1.1), samples),
        generator.uniform(*(np.asarray(surface.z_bounds) * 1.1), samples)])
    points, gradient = surface_embedding(surface, spread)
    numerical = np.column_stack([
        (surface.evaluate(spread + offset)[:, 1] - surface.evaluate(spread - offset)[:, 1]) / (2 * step)
        for offset in (np.array([step, 0.0]), np.array([0.0, step]))])

    flat = type(surface)(surface.center, surface.scale, np.zeros((4, 4)), surface.x_bounds,
                         surface.z_bounds, surface.origin, surface.config)
    corners = np.column_stack([surface.x_bounds, surface.z_bounds]) * 0.8
    plane_length, plane_chord, _ = geodesic(flat, corners[0], corners[1])
    return {
        'evaluate max abs difference': float(np.abs(points - surface.evaluate(spread)).max()),
        'gradient max abs difference': float(np.abs(gradient - numerical).max()),
        'plane geodesic - chord': float(plane_length - plane_chord),
    }


directories = ([path for path in sorted(ARTIFACT_ROOT.iterdir())] if MODELS is None
               else [ARTIFACT_ROOT / name for name in MODELS])
bundles = {}
for directory in directories:
    if not (directory / TRAINING_ROWS).is_file():
        continue
    _, metadata, surface = load_surface_bundle(directory / 'surface')
    bundles[directory.name] = dict(directory=directory, surface=surface, metadata=metadata)
if not bundles:
    raise ValueError(f'No model directories with {TRAINING_ROWS} under {ARTIFACT_ROOT}')

checks = pd.DataFrame({name: solver_checks(bundle['surface'])
                       for name, bundle in bundles.items()}).T
checks.index.name = 'model'
display(checks)
assert checks['evaluate max abs difference'].max() < 1e-12, 'Embedding disagrees with the saved surface.'
assert checks['gradient max abs difference'].max() < 1e-6, 'Analytic gradient is wrong.'
assert checks['plane geodesic - chord'].abs().max() < 1e-9, 'A plane geodesic is not straight.'
print(f'{len(bundles)} surfaces loaded and checked.')

,evaluate max abs difference,gradient max abs difference,plane geodesic - chord
model,,,
gemma-4-31B-it,0.000000e+00,3.656498e-08,0.000000e+00
Qwen3-14B,0.000000e+00,1.835807e-08,-1.421085e-14
Qwen3-32B,3.552714e-15,1.175057e-08,0.000000e+00
Qwen3-4B-Instruct-2507,0.000000e+00,9.968108e-09,-7.105427e-15
Qwen3-8B,0.000000e+00,4.624478e-08,-1.421085e-14


5 surfaces loaded and checked.


## Token lengths

Counted with the model's own tokenizer, reproducing what `scripts/cache_activations.py` fed
the model through `ChatTemplateTokenizer`: an empty system prompt, thinking disabled, a
generation prompt appended, and the formatted string then encoded without a second round of
special tokens.

Fetching the five tokenizers takes a few seconds each the first time and is cached
afterwards; no model weights are downloaded.

In [4]:
def chat_text(tokenizer, texts):
    """The formatted strings cache_activations.py would have tokenized."""
    return [tokenizer.apply_chat_template([{'role': 'user', 'content': text}], tokenize=False,
                                          add_special_tokens=True, add_generation_prompt=True,
                                          enable_thinking=False)
            for text in texts]


def token_lengths(model_name, texts):
    """Token count per prompt, plus the constant the chat template adds."""
    from transformers import AutoTokenizer

    cache_dir = None if HF_CACHE_DIR is None else str(HF_CACHE_DIR)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, padding_side='left',
                                              cache_dir=cache_dir)
    bare = np.array([len(ids) for ids in tokenizer(texts, add_special_tokens=False)['input_ids']])
    if TOKEN_TEXT == 'bare':
        return bare, 0
    formatted = np.array([len(ids) for ids in
                          tokenizer(chat_text(tokenizer, texts),
                                    add_special_tokens=False)['input_ids']])
    overhead = np.unique(formatted - bare)
    if len(overhead) != 1:
        raise ValueError(f'{model_name}: chat template overhead is not constant ({overhead}).')
    return formatted, int(overhead[0])

## One table per model

For each variant: the two class centroids, the geodesic between them, the same geodesic
re-solved at `REFINEMENT_NODES` as a convergence check, and the variant's mean prompt
length. The class ordering is read from each model's own rows and the configured anchors
are checked against it, so a pair that ran backwards in some model fails here rather than
quietly producing a distance with the wrong meaning.

In [5]:
def check_anchor_ordering(rows):
    """The model's own class ordering - ascending mean bcpc_arc_length, as the pipeline does."""
    order = rows.groupby(STAKES)[TARGET].mean().sort_values(kind='stable').index.tolist()
    absent = [name for name in (LOW_STAKES_CLASS, HIGH_STAKES_CLASS) if name not in order]
    if absent:
        raise ValueError(f'Configured stakes classes absent from the training rows: {absent}')
    if order.index(LOW_STAKES_CLASS) >= order.index(HIGH_STAKES_CLASS):
        raise ValueError(f'{LOW_STAKES_CLASS} does not sit below {HIGH_STAKES_CLASS}: {order}')
    return order


def variant_table(bundle):
    surface = bundle['surface']
    rows = pd.read_parquet(bundle['directory'] / TRAINING_ROWS,
                           columns=[VARIANT, STAKES, TEXT, TARGET, SURFACE_U, SURFACE_V])
    ordering = check_anchor_ordering(rows)
    rows['tokens'], overhead = token_lengths(bundle['metadata']['model_name'], rows[TEXT].tolist())

    counted = rows if TOKEN_SCOPE == 'all' else rows.loc[
        rows[STAKES].isin([LOW_STAKES_CLASS, HIGH_STAKES_CLASS])]
    lengths = counted.groupby(VARIANT)['tokens'].agg(['mean', 'min', 'max', 'size'])
    positions = rows.groupby([VARIANT, STAKES])[[SURFACE_U, SURFACE_V]].mean()
    counts = rows.groupby([VARIANT, STAKES]).size()

    records = []
    for variant in sorted(rows[VARIANT].unique()):
        for stakes in (LOW_STAKES_CLASS, HIGH_STAKES_CLASS):
            if (variant, stakes) not in positions.index:
                raise ValueError(f'{variant} has no {stakes} rows; its centroid is undefined.')
        low, high = (positions.loc[(variant, stakes)].to_numpy(dtype=np.float64)
                     for stakes in (LOW_STAKES_CLASS, HIGH_STAKES_CLASS))
        length, chord, path = geodesic(surface, low, high)
        refined, _, _ = geodesic(surface, low, high, nodes=REFINEMENT_NODES)
        outside = int(((path[:, 0] < surface.x_bounds[0])
                       | (path[:, 0] > surface.x_bounds[1])).sum())
        records.append({
            'variant': variant,
            'register': variant.split('--')[0],
            'template': variant.split('--')[1].rsplit('-', 1)[0],
            'mean_tokens': float(lengths.loc[variant, 'mean']),
            'min_tokens': int(lengths.loc[variant, 'min']),
            'max_tokens': int(lengths.loc[variant, 'max']),
            'prompts_counted': int(lengths.loc[variant, 'size']),
            'geodesic': length,
            'chord': chord,
            'curvature_ratio': length / chord,
            'refinement_shift': abs(refined - length),
            'nodes_outside_fit': outside,
            f'{LOW_STAKES_CLASS}_rows': int(counts.loc[(variant, LOW_STAKES_CLASS)]),
            f'{HIGH_STAKES_CLASS}_rows': int(counts.loc[(variant, HIGH_STAKES_CLASS)]),
        })
    # The row frame carries the tokenizer counts the matched control below reuses.
    return pd.DataFrame(records), rows.drop(columns=[TEXT]), ordering, overhead


tables = {}
for name, bundle in bundles.items():
    started = time.time()
    frame, row_frame, ordering, overhead = variant_table(bundle)
    tables[name] = dict(frame=frame, rows=row_frame, ordering=ordering,
                        overhead=overhead, **bundle)
    print(f'{name:24s} {len(frame):3d} variants  '
          f'{LOW_STAKES_CLASS} -> {HIGH_STAKES_CLASS} at positions '
          f'{ordering.index(LOW_STAKES_CLASS)} and {ordering.index(HIGH_STAKES_CLASS)} of '
          f'{len(ordering)}  geodesic {frame.geodesic.min():.2f}-{frame.geodesic.max():.2f}  '
          f'tokens {frame.mean_tokens.min():.1f}-{frame.mean_tokens.max():.1f} '
          f'(template adds {overhead})  {time.time() - started:.1f}s')

c:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


gemma-4-31B-it            41 variants  very_low -> catastrophic at positions 0 and 7 of 8  geodesic 29.02-97.47  tokens 25.9-82.9 (template adds 13)  79.0s
Qwen3-14B                 41 variants  very_low -> catastrophic at positions 0 and 6 of 8  geodesic 22.81-76.26  tokens 25.9-78.9 (template adds 12)  29.9s
Qwen3-32B                 41 variants  very_low -> catastrophic at positions 0 and 6 of 8  geodesic 22.57-74.16  tokens 25.9-78.9 (template adds 12)  29.1s
Qwen3-4B-Instruct-2507    41 variants  very_low -> catastrophic at positions 0 and 6 of 8  geodesic 10.03-33.87  tokens 21.9-74.9 (template adds 8)  38.7s
Qwen3-8B                  41 variants  very_low -> catastrophic at positions 0 and 6 of 8  geodesic 16.61-43.54  tokens 25.9-78.9 (template adds 12)  15.3s


### Numerical diagnostics

`worst refinement shift` is how far a distance moved when its path was re-solved at double
the node count - the discretization error of the reported number. `curvature ratio` is the
geodesic over the ambient chord: what the curvature of the surface is worth here, and never
below 1. `path nodes outside fit` counts nodes that landed on the surface's linear extension
beyond the fitted PLS1 range, where the geometry is an extrapolation.

In [6]:
diagnostics = pd.DataFrame({
    name: {
        'variants': len(entry['frame']),
        'worst refinement shift': entry['frame'].refinement_shift.max(),
        'worst relative shift': (entry['frame'].refinement_shift / entry['frame'].geodesic).max(),
        'curvature ratio': f"{entry['frame'].curvature_ratio.min():.4f} to "
                           f"{entry['frame'].curvature_ratio.max():.4f}",
        'geodesic over chord by': f"{(entry['frame'].geodesic - entry['frame'].chord).max():.3f} max",
        'path nodes outside fit': int(entry['frame'].nodes_outside_fit.sum()),
    } for name, entry in tables.items()}).T
diagnostics.index.name = 'model'
display(diagnostics)
assert max(entry['frame'].refinement_shift.max() for entry in tables.values()) < 1e-2, \
    'Distances have not converged in the node count; raise GEODESIC_NODES.'

,variants,worst refinement shift,worst relative shift,curvature ratio,geodesic over chord by,path nodes outside fit
model,,,,,,
gemma-4-31B-it,41,0.006289,0.000065,1.0448 to 1.4108,27.742 max,0
Qwen3-14B,41,0.002245,0.000035,1.0053 to 1.1797,11.617 max,0
Qwen3-32B,41,0.000876,0.000012,1.0004 to 1.0157,1.147 max,0
Qwen3-4B-Instruct-2507,41,0.001241,0.000037,1.0419 to 1.2615,7.021 max,0
Qwen3-8B,41,0.000216,0.000005,1.0000 to 1.0266,1.128 max,0


## The panels

One scatter per model, 41 variants each, with the least-squares line through them and that
model's Pearson `r` and Spearman `rho`. Both are reported because the relationship need not
be linear and there are only 41 points: `r` measures the line that is drawn, `rho` measures
whether the ordering holds at all. Axes are per panel; the y units are not comparable
between models.

In [7]:
SURFACE = '#ffffff'
SERIES = '#2a78d6'          # categorical slot 1; a single series needs no legend
GRID = '#e6e5e1'
INK_PRIMARY, INK_SECONDARY, INK_MUTED = '#0b0b0b', '#52514e', '#8a8983'
COLUMNS = 3


def fit_line(x, y, pad=0.04):
    slope, intercept = np.polyfit(x, y, 1)
    span = np.array([x.min(), x.max()])
    span = span + pad * (span[1] - span[0]) * np.array([-1.0, 1.0])
    return span, intercept + slope * span, slope


def draw_model(figure, entry, row, column):
    frame = entry['frame']
    x, y = frame.mean_tokens.to_numpy(), frame.geodesic.to_numpy()
    span, fitted, slope = fit_line(x, y)
    figure.add_trace(go.Scatter(x=span, y=fitted, mode='lines', showlegend=False,
                                line=dict(color=INK_MUTED, width=2, dash='dot'),
                                hoverinfo='skip'), row=row, col=column)
    figure.add_trace(go.Scatter(
        x=x, y=y, mode='markers', showlegend=False,
        customdata=np.column_stack([frame.register, frame.template, frame.curvature_ratio]),
        marker=dict(color=SERIES, size=9, line=dict(color=SURFACE, width=2)),
        hovertemplate='%{customdata[0]} / %{customdata[1]}<br>'
                      '%{x:.1f} tokens<br>geodesic %{y:.2f}'
                      '<br>%{customdata[2]:.3f} x the chord<extra></extra>'),
        row=row, col=column)
    r, r_p = pearsonr(x, y)
    rho, rho_p = spearmanr(x, y)
    figure.add_annotation(
        text=(f'r = {r:+.2f} (p = {r_p:.3f})<br>rho = {rho:+.2f} (p = {rho_p:.3f})'
              f'<br>slope {slope:+.3f} per token'),
        xref='x domain', yref='y domain', x=0.03, y=0.97, xanchor='left', yanchor='top',
        showarrow=False, align='left', font=dict(size=10, color=INK_SECONDARY),
        row=row, col=column)


names = list(tables)
row_count = int(np.ceil(len(names) / COLUMNS))
figure = make_subplots(rows=row_count, cols=COLUMNS, subplot_titles=names,
                       horizontal_spacing=0.075, vertical_spacing=0.16)
for index, name in enumerate(names):
    row, column = divmod(index, COLUMNS)
    draw_model(figure, tables[name], row + 1, column + 1)
    figure.update_xaxes(title_text='mean prompt length (tokens)', row=row + 1, col=column + 1)
    if column == 0:
        figure.update_yaxes(title_text='geodesic distance (PLS score units)',
                            row=row + 1, col=column + 1)
for index in range(len(names), row_count * COLUMNS):       # blank the unused grid cells
    row, column = divmod(index, COLUMNS)
    figure.update_xaxes(visible=False, row=row + 1, col=column + 1)
    figure.update_yaxes(visible=False, row=row + 1, col=column + 1)

figure.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=GRID,
                    title_font=dict(size=11, color=INK_SECONDARY),
                    tickfont=dict(size=10, color=INK_SECONDARY))
figure.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID,
                    title_font=dict(size=11, color=INK_SECONDARY),
                    tickfont=dict(size=10, color=INK_SECONDARY))
for annotation in figure.layout.annotations[:len(names)]:
    annotation.font.update(size=12, color=INK_PRIMARY)
figure.add_annotation(
    text=f'One point per variant, {len(tables[names[0]]["frame"])} per model. The geodesic runs '
         f'{LOW_STAKES_CLASS} centroid to {HIGH_STAKES_CLASS} centroid along that model\'s '
         f'fitted surface; every panel keeps its own scale.',
    xref='paper', yref='paper', x=0, y=1.055, xanchor='left', yanchor='bottom',
    showarrow=False, align='left', font=dict(size=11, color=INK_SECONDARY))
figure.update_layout(template='plotly_white', height=760,
                     title=dict(text='Anchor-class separation against prompt length',
                                x=0, xanchor='left', font=dict(size=16, color=INK_PRIMARY)),
                     paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
                     margin=dict(t=135, l=75, r=30, b=60),
                     hoverlabel=dict(bgcolor=SURFACE, bordercolor=GRID,
                                     font=dict(size=11, color=INK_PRIMARY)))
figure.show()

### The same numbers as a table

The panels read by eye; this is what they are made of. `r^2` is the share of the
between-variant spread in the geodesic that a straight line in prompt length accounts for.

In [8]:
def correlation_row(frame):
    x, y = frame.mean_tokens.to_numpy(), frame.geodesic.to_numpy()
    r, r_p = pearsonr(x, y)
    rho, rho_p = spearmanr(x, y)
    slope, _ = np.polyfit(x, y, 1)
    return {
        'variants': len(frame),
        'tokens': f'{x.min():.1f} to {x.max():.1f}',
        'geodesic': f'{y.min():.2f} to {y.max():.2f}',
        'geodesic spread / median': (y.max() - y.min()) / float(np.median(y)),
        'pearson r': r,
        'r p-value': r_p,
        'r^2': r ** 2,
        'spearman rho': rho,
        'rho p-value': rho_p,
        'slope per token': slope,
    }


correlations = pd.DataFrame({name: correlation_row(entry['frame'])
                             for name, entry in tables.items()}).T
correlations.index.name = 'model'
display(correlations)

,variants,tokens,geodesic,geodesic spread / median,pearson r,r p-value,r^2,spearman rho,rho p-value,slope per token
model,,,,,,,,,,
gemma-4-31B-it,41,25.9 to 82.9,29.02 to 97.47,0.949178,0.065235,0.685316,0.004256,-0.082679,0.607314,0.076368
Qwen3-14B,41,25.9 to 78.9,22.81 to 76.26,1.071369,-0.599021,0.000035,0.358826,-0.683686,0.000001,-0.566465
Qwen3-32B,41,25.9 to 78.9,22.57 to 74.16,1.191715,-0.59548,0.00004,0.354597,-0.512895,0.000606,-0.576955
Qwen3-4B-Instruct-2507,41,21.9 to 74.9,10.03 to 33.87,1.063173,-0.477639,0.001588,0.228139,-0.599207,0.000035,-0.210064
Qwen3-8B,41,25.9 to 78.9,16.61 to 43.54,1.00297,-0.504638,0.000767,0.25466,-0.538466,0.000282,-0.275993


### Is it a register effect?

Prompt length is largely a property of the register - `verbose_no_horizon` prompts are long
and `task_only` ones are short - so a correlation across all 41 variants could be six
register clusters in a row rather than a trend within any of them. The first table is each
register's mean on both axes; the second re-runs the correlation *within* each register,
where length varies only by template wording.

In [9]:
by_register = pd.concat(
    {name: entry['frame'].groupby('register')[['mean_tokens', 'geodesic']].agg(['mean', 'size'])
     for name, entry in tables.items()}, names=['model'])
by_register.columns = ['mean tokens', 'variants', 'mean geodesic', '_drop']
print('Register means')
display(by_register.drop(columns='_drop').round(2))

within = []
for name, entry in tables.items():
    for register, group in entry['frame'].groupby('register'):
        if len(group) < 4:
            continue
        r, p = pearsonr(group.mean_tokens, group.geodesic)
        within.append({'model': name, 'register': register, 'variants': len(group),
                       'tokens': f'{group.mean_tokens.min():.1f} to {group.mean_tokens.max():.1f}',
                       'pearson r': round(r, 3), 'p-value': round(p, 3)})
print('\nWithin-register correlation (registers with at least four variants)')
display(pd.DataFrame(within).set_index(['model', 'register']))

Register means


mean tokens  variants  \
model                  register                                        
gemma-4-31B-it         conversational_no_time        28.89         3   
                       directive_no_horizon          27.22         6   
                       expert_no_horizon             38.22         6   
                       impersonal_no_horizon         33.72         6   
                       task_only                     35.39        12   
                       verbose_no_horizon            66.64         8   
Qwen3-14B              conversational_no_time        27.93         3   
                       directive_no_horizon          26.43         6   
                       expert_no_horizon             37.26         6   
                       impersonal_no_horizon         33.09         6   
                       task_only                     34.09        12   
                       verbose_no_horizon            64.68         8   
Qwen3-32B              conversational_no_time        27.93         3   
                       directive_no_horizon          26.43         6   
                       expert_no_horizon             37.26         6   
                       impersonal_no_horizon         33.09         6   
                       task_only                     34.09        12   
                       verbose_no_horizon            64.68         8   
Qwen3-4B-Instruct-2507 conversational_no_time        23.93         3   
                       directive_no_horizon          22.43         6   
                       expert_no_horizon             33.26         6   
                       impersonal_no_horizon         29.09         6   
                       task_only                     30.09        12   
                       verbose_no_horizon            60.68         8   
Qwen3-8B               conversational_no_time        27.93         3   
                       directive_no_horizon          26.43         6   
                       expert_no_horizon             37.26         6   
                       impersonal_no_horizon         33.09         6   
                       task_only                     34.09        12   
                       verbose_no_horizon            64.68         8   

                                               mean geodesic  
model                  register                               
gemma-4-31B-it         conversational_no_time          72.65  
                       directive_no_horizon            72.13  
                       expert_no_horizon               42.08  
                       impersonal_no_horizon           58.84  
                       task_only                       81.12  
                       verbose_no_horizon              70.94  
Qwen3-14B              conversational_no_time          55.73  
                       directive_no_horizon            56.96  
                       expert_no_horizon               37.66  
                       impersonal_no_horizon           50.04  
                       task_only                       54.78  
                       verbose_no_horizon              34.45  
Qwen3-32B              conversational_no_time          42.94  
                       directive_no_horizon            49.15  
                       expert_no_horizon               39.30  
                       impersonal_no_horizon           42.17  
                       task_only                       61.84  
                       verbose_no_horizon              30.44  
Qwen3-4B-Instruct-2507 conversational_no_time          26.72  
                       directive_no_horizon            25.68  
                       expert_no_horizon               19.30  
                       impersonal_no_horizon           18.54  
                       task_only                       24.19  
                       verbose_no_horizon              16.71  
Qwen3-8B               conversational_no_time          28.65  
                       directive_no_horiz


Within-register correlation (registers with at least four variants)


variants        tokens  \
model                  register                                        
gemma-4-31B-it         directive_no_horizon          6  25.9 to 28.9   
                       expert_no_horizon             6  33.9 to 41.9   
                       impersonal_no_horizon         6  27.9 to 35.9   
                       task_only                    12  31.9 to 39.9   
                       verbose_no_horizon            8  54.9 to 82.9   
Qwen3-14B              directive_no_horizon          6  25.9 to 27.9   
                       expert_no_horizon             6  32.9 to 40.9   
                       impersonal_no_horizon         6  26.9 to 35.9   
                       task_only                    12  29.9 to 38.9   
                       verbose_no_horizon            8  53.9 to 78.9   
Qwen3-32B              directive_no_horizon          6  25.9 to 27.9   
                       expert_no_horizon             6  32.9 to 40.9   
                       impersonal_no_horizon         6  26.9 to 35.9   
                       task_only                    12  29.9 to 38.9   
                       verbose_no_horizon            8  53.9 to 78.9   
Qwen3-4B-Instruct-2507 directive_no_horizon          6  21.9 to 23.9   
                       expert_no_horizon             6  28.9 to 36.9   
                       impersonal_no_horizon         6  22.9 to 31.9   
                       task_only                    12  25.9 to 34.9   
                       verbose_no_horizon            8  49.9 to 74.9   
Qwen3-8B               directive_no_horizon          6  25.9 to 27.9   
                       expert_no_horizon             6  32.9 to 40.9   
                       impersonal_no_horizon         6  26.9 to 35.9   
                       task_only                    12  29.9 to 38.9   
                       verbose_no_horizon            8  53.9 to 78.9   

                                              pearson r  p-value  
model                  register                                   
gemma-4-31B-it         directive_no_horizon      -0.553    0.255  
                       expert_no_horizon         -0.136    0.797  
                       impersonal_no_horizon     -0.282    0.588  
                       task_only                 -0.236    0.461  
                       verbose_no_horizon         0.681    0.063  
Qwen3-14B              directive_no_horizon      -0.405    0.426  
                       expert_no_horizon         -0.134    0.800  
                       impersonal_no_horizon     -0.593    0.215  
                       task_only                 -0.694    0.012  
                       verbose_no_horizon        -0.077    0.856  
Qwen3-32B              directive_no_horizon      -0.740    0.093  
                       expert_no_horizon         -0.056    0.916  
                       impersonal_no_horizon     -0.705    0.118  
                       task_only                 -0.729    0.007  
                       verbose_no_horizon        -0.675    0.066  
Qwen3-4B-Instruct-2507 directive_no_horizon      -0.450    0.371  
                       expert_no_horizon         -0.190    0.719  
                       impersonal_no_horizon     -0.769    0.074  
                       task_only                 -0.634    0.027  
                       verbose_no_horizon         0.172    0.683  
Qwen3-8B               directive_no_horizon      -0.708    0.116  
                       expert_no_horizon         -0.141    0.790  
                       impersonal_no_horizon     -0.576    0.231  
                       task_only                 -0.601    0.039  
                       verbose_no_horizon        -0.644    0.085

## Does the coordinate survive matching on length?

Everything above asks whether a variant's *span* moves with its prompt length. A more basic
question sits underneath it: is `arc_length_parallel` a length readout at all? A correlation
cannot settle that, because across the corpus length and stakes vary at once. Matching can.
Compare `very_low` against `catastrophic` only between prompts of **identical token length**
and length is held exactly fixed, so whatever separation survives is not something length
could have produced.

The estimator is the usual one for exact matching. Within each token count both classes
reach, take each class's mean coordinate; weight that count by the smaller of its two cell
sizes, so a length carried by forty `catastrophic` rows and two `very_low` ones contributes
the two pairs it can support rather than forty; then average the per-count differences with
those weights. Token counts only one class reaches are dropped, and `matched_pairs` reports
how much of the corpus survives that restriction. `within_variant_gap` matches inside a
variant as well, so no comparison crosses a template - a stricter control on thinner support.

Two further numbers make the reading unambiguous.

- `length_attributable` multiplies the **within-class** slope of the coordinate on token
  length by the between-class token gap: what a length effect of the size actually present
  in this data would contribute to the anchor separation. It belongs beside the raw gap it
  is offered to explain, and it carries its own sign, which need not be the helpful one.
- `class_mean_r` correlates the eight class-mean token counts against the eight class-mean
  coordinates. It is here because it is the number a sceptical reader reaches for first,
  because it looks alarming, and because it is an aggregation artifact: eight points over a
  two-token range, with a within-class slope of the opposite sign.

In [ ]:
def matched_gap(rows, column, within=None, bin_width=None):
    """Exact-matching estimate of the anchor gap, with token length held fixed.

    `within` names further columns to match inside - a variant, say - so that no
    comparison crosses a template. Returns the estimate and the number of matched
    pairs backing it, which is the sum of the per-bin smaller cell sizes.
    """
    bin_width = TOKEN_MATCH_BIN if bin_width is None else bin_width
    pair = rows.loc[rows[STAKES].isin([LOW_STAKES_CLASS, HIGH_STAKES_CLASS])].copy()
    pair['token_bin'] = (pair['tokens'] // bin_width) * bin_width
    keys = [*(within or []), 'token_bin']
    means = pair.groupby([*keys, STAKES], observed=True)[column].mean().unstack()
    sizes = pair.groupby([*keys, STAKES], observed=True).size().unstack()
    anchors = [LOW_STAKES_CLASS, HIGH_STAKES_CLASS]
    if not set(anchors) <= set(means.columns):
        raise ValueError('An anchor class is absent from the rows being matched.')
    keep = means[anchors].notna().all(axis=1)
    if not keep.any():
        raise ValueError('No token length is reached by both anchor classes.')
    weights = sizes.loc[keep, anchors].min(axis=1)
    difference = means.loc[keep, HIGH_STAKES_CLASS] - means.loc[keep, LOW_STAKES_CLASS]
    return float((difference * weights).sum() / weights.sum()), int(weights.sum())


def matching_row(rows):
    """One model's raw gap, its matched counterpart, and what length could explain."""
    pair = rows.loc[rows[STAKES].isin([LOW_STAKES_CLASS, HIGH_STAKES_CLASS])]
    means = pair.groupby(STAKES, observed=True)[[TARGET, 'tokens']].mean()
    raw = float(means.loc[HIGH_STAKES_CLASS, TARGET] - means.loc[LOW_STAKES_CLASS, TARGET])
    token_gap = float(means.loc[HIGH_STAKES_CLASS, 'tokens'] - means.loc[LOW_STAKES_CLASS, 'tokens'])
    # The slope is taken within class, where it is not contaminated by the ladder itself.
    centred_tokens = (rows['tokens'] - rows.groupby(STAKES, observed=True)['tokens']
                      .transform('mean')).to_numpy(dtype=np.float64)
    centred_target = (rows[TARGET] - rows.groupby(STAKES, observed=True)[TARGET]
                      .transform('mean')).to_numpy(dtype=np.float64)
    slope = float(centred_tokens @ centred_target / (centred_tokens @ centred_tokens))
    matched, pairs = matched_gap(rows, TARGET)
    inside, inside_pairs = matched_gap(rows, TARGET, within=[VARIANT])
    class_means = rows.groupby(STAKES, observed=True)[['tokens', TARGET]].mean()
    return {
        'raw_gap': raw,
        'matched_gap': matched,
        'retained': matched / raw,
        'matched_pairs': pairs,
        'within_variant_gap': inside,
        'within_variant_pairs': inside_pairs,
        'token_gap': token_gap,
        'slope_per_token': slope,
        'length_attributable': slope * token_gap,
        'class_mean_r': float(pearsonr(class_means['tokens'], class_means[TARGET])[0]),
    }


matching = pd.DataFrame({name: matching_row(entry['rows']) for name, entry in tables.items()}).T
matching.index.name = 'model'
display(matching.astype(float).round(4))
share = (matching.length_attributable / matching.raw_gap).abs().max()
print(f'Length accounts for at most {100 * share:.2f}% of any model\'s anchor gap; '
      f'matching on length retains {100 * matching.retained.min():.1f}% to '
      f'{100 * matching.retained.max():.1f}% of it.')

### The same control on the geodesic

The table above matches the coordinate, but the panels plot a geodesic between centroids, so
the control has to be rebuilt at that level before it can speak to them. `matched_centroids`
averages each class's surface position over the token counts both classes reach, with the
same smaller-cell weights, and the geodesic is then solved between those two matched
centroids by the solver used everywhere else in this notebook.

Matching inside a single variant leaves only about twenty pairs behind each panel point, so
these are thinner estimates than the raw ones and some of the gap between the two columns is
that thinness rather than any length effect. `matched_bins` reports the support directly.

`r matched` is this notebook's headline correlation recomputed on matched centroids. If the
relationship the panels report were a length artifact, it would collapse here.

In [ ]:
def matched_centroids(rows, bin_width=None):
    """Class centroids built only from the token lengths both classes reach.

    Each surviving token count contributes its two class means weighted by the smaller
    of its two cell sizes, so the two centroids are averaged over one common length
    distribution rather than over the two the classes happen to have.
    """
    bin_width = TOKEN_MATCH_BIN if bin_width is None else bin_width
    anchors = [LOW_STAKES_CLASS, HIGH_STAKES_CLASS]
    pair = rows.loc[rows[STAKES].isin(anchors)].copy()
    pair['token_bin'] = (pair['tokens'] // bin_width) * bin_width
    means = pair.groupby(['token_bin', STAKES], observed=True)[[SURFACE_U, SURFACE_V]].mean()
    sizes = pair.groupby(['token_bin', STAKES], observed=True).size().unstack()
    if not set(anchors) <= set(sizes.columns):
        raise ValueError('An anchor class is absent from the rows being matched.')
    keep = sizes[anchors].notna().all(axis=1)
    if not keep.any():
        raise ValueError('No token length is reached by both anchor classes.')
    bins = sizes.index[keep]
    weights = sizes.loc[bins, anchors].min(axis=1).to_numpy(dtype=np.float64)
    centroid = lambda stakes: np.average(
        np.array([means.loc[(token_bin, stakes)].to_numpy(dtype=np.float64) for token_bin in bins]),
        axis=0, weights=weights)
    return centroid(LOW_STAKES_CLASS), centroid(HIGH_STAKES_CLASS), int(weights.sum()), len(bins)


def matched_geodesics(entry):
    """Per variant: the raw geodesic of the panels, and its length-matched counterpart."""
    rows, surface = entry['rows'], entry['surface']
    records = []
    for variant in sorted(rows[VARIANT].unique()):
        group = rows.loc[rows[VARIANT] == variant]
        low, high, pairs, bins = matched_centroids(group)
        matched, _, _ = geodesic(surface, low, high)
        records.append({'variant': variant, 'matched_geodesic': matched,
                        'matched_pairs': pairs, 'matched_bins': bins})
    return entry['frame'].merge(pd.DataFrame(records), on='variant', validate='one_to_one')


matched_tables, rows_out = {}, {}
for name, entry in tables.items():
    frame = matched_geodesics(entry)
    matched_tables[name] = frame
    raw_r, _ = pearsonr(frame.mean_tokens, frame.geodesic)
    matched_r, matched_p = pearsonr(frame.mean_tokens, frame.matched_geodesic)
    rows_out[name] = {
        'variants': len(frame),
        'matched pairs per variant': f'{frame.matched_pairs.min()} to {frame.matched_pairs.max()}',
        'matched bins per variant': f'{frame.matched_bins.min()} to {frame.matched_bins.max()}',
        'matched / raw, median': (frame.matched_geodesic / frame.geodesic).median(),
        'matched / raw, worst': (frame.matched_geodesic / frame.geodesic - 1).abs().max() + 1,
        'r raw': raw_r,
        'r matched': matched_r,
        'p matched': matched_p,
    }
matched_summary = pd.DataFrame(rows_out).T
matched_summary.index.name = 'model'
display(matched_summary)


figure = make_subplots(rows=row_count, cols=COLUMNS, subplot_titles=names,
                       horizontal_spacing=0.075, vertical_spacing=0.16)
for index, name in enumerate(names):
    row, column = divmod(index, COLUMNS)
    frame = matched_tables[name]
    limits = np.array([min(frame.geodesic.min(), frame.matched_geodesic.min()),
                       max(frame.geodesic.max(), frame.matched_geodesic.max())])
    limits = limits + 0.06 * (limits[1] - limits[0]) * np.array([-1.0, 1.0])
    figure.add_trace(go.Scatter(x=limits, y=limits, mode='lines', showlegend=False,
                                line=dict(color=INK_MUTED, width=2, dash='dot'),
                                hoverinfo='skip'), row=row + 1, col=column + 1)
    figure.add_trace(go.Scatter(
        x=frame.geodesic, y=frame.matched_geodesic, mode='markers', showlegend=False,
        customdata=np.column_stack([frame.register, frame.template, frame.matched_pairs]),
        marker=dict(color=SERIES, size=9, line=dict(color=SURFACE, width=2)),
        hovertemplate='%{customdata[0]} / %{customdata[1]}<br>raw %{x:.2f}'
                      '<br>matched %{y:.2f}<br>%{customdata[2]} matched pairs<extra></extra>'),
        row=row + 1, col=column + 1)
    figure.update_xaxes(title_text='geodesic (all rows)', range=list(limits), row=row + 1, col=column + 1)
    figure.update_yaxes(title_text='geodesic (length-matched)' if column == 0 else None,
                        range=list(limits), row=row + 1, col=column + 1)
for index in range(len(names), row_count * COLUMNS):
    row, column = divmod(index, COLUMNS)
    figure.update_xaxes(visible=False, row=row + 1, col=column + 1)
    figure.update_yaxes(visible=False, row=row + 1, col=column + 1)
figure.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=GRID,
                    title_font=dict(size=11, color=INK_SECONDARY),
                    tickfont=dict(size=10, color=INK_SECONDARY))
figure.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID,
                    title_font=dict(size=11, color=INK_SECONDARY),
                    tickfont=dict(size=10, color=INK_SECONDARY))
for annotation in figure.layout.annotations[:len(names)]:
    annotation.font.update(size=12, color=INK_PRIMARY)
figure.add_annotation(
    text='Each point is a variant; the dotted line is equality. The matched geodesic uses only '
         'the token lengths both anchor classes reach, so a point that sits on the line is a '
         'span that prompt length cannot account for.',
    xref='paper', yref='paper', x=0, y=1.055, xanchor='left', yanchor='bottom',
    showarrow=False, align='left', font=dict(size=11, color=INK_SECONDARY))
figure.update_layout(template='plotly_white', height=760,
                     title=dict(text='Anchor separation before and after matching on prompt length',
                                x=0, xanchor='left', font=dict(size=16, color=INK_PRIMARY)),
                     paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
                     margin=dict(t=135, l=75, r=30, b=60),
                     hoverlabel=dict(bgcolor=SURFACE, bordercolor=GRID,
                                     font=dict(size=11, color=INK_PRIMARY)))
figure.show()

## Optional: write the per-variant tables

Off by default. Set `WRITE_TABLE = True` in the configuration cell to write each model's
41-row table to
`artifacts/content/artifacts/<model>/surface/anchor_geodesic_vs_token_length.csv`. Nothing
already on disk is touched; the file is new, and rewritten whole on each run.

In [10]:
if WRITE_TABLE:
    for name, entry in tables.items():
        path = entry['directory'] / 'surface' / TABLE_NAME
        entry['frame'].assign(model=name, model_name=entry['metadata']['model_name'],
                              low_class=LOW_STAKES_CLASS, high_class=HIGH_STAKES_CLASS,
                              token_text=TOKEN_TEXT, token_scope=TOKEN_SCOPE,
                              geodesic_nodes=GEODESIC_NODES).to_csv(path, index=False)
        print(f'Wrote {path.relative_to(ROOT)} ({len(entry["frame"])} rows)')
else:
    print('WRITE_TABLE is False; nothing written.')

WRITE_TABLE is False; nothing written.
